# 1
a = np.arange(6).reshape(2, 3); print(a[1, 2], a.shape)
# 2
a = np.array([3, 8, 1, 8]); print(a.argmax(), a.max())
# 3
a = np.arange(10); b = a[::3]; b[0] = 99; print(a[0])
# 4
a = np.array([1, 5, 7, 2]); print((a > 3).sum(), (a > 3).mean())
# 5
print(np.zeros((2, 3)).shape, np.linspace(0, 1, 5)[1])

Мои предсказания и объяснения:
1. 5 (2, 3) — во второй строке последний элемент 5, форма две строки и три столбца.
2. 1 8 — argmax() возвращает индекс первого максимума, а max() — само число.
3. 99 — срез NumPy является представлением исходного массива, поэтому меняется a.
4. 2 0.5 — два элемента больше 3, то есть половина массива.
5. (2, 3) 0.25 — матрица из двух строк и трёх столбцов; второй элемент пяти равномерных точек от 0 до 1 равен 0.25.

In [7]:
# Проверяем предсказания
import numpy as np
a=np.arange(6).reshape(2,3)
print('1:',a[1,2],a.shape)
a=np.array([3,8,1,8])
print('2:',a.argmax(),a.max())
a=np.arange(10)
b=a[::3]
b[0]=99
print('3:',a[0])
a=np.array([1,5,7,2])
print('4:',(a>3).sum(),(a>3).mean())
print('5:',np.zeros((2,3)).shape,np.linspace(0,1,5)[1])


1: 5 (2, 3)
2: 1 8
3: 99
4: 2 0.5
5: (2, 3) 0.25


In [8]:
import csv

with open("orders.csv", encoding="utf-8-sig") as f:
    rows = list(csv.DictReader(f))

In [9]:
import csv
import numpy as np

CITY = {"алматы": "Алматы", "almaty": "Алматы", "астана": "Астана", "astana": "Астана",
        "шымкент": "Шымкент", "shymkent": "Шымкент", "караганда": "Караганда",
        "актобе": "Актобе", "атырау": "Атырау"}

with open("orders.csv", encoding="utf-8") as f:
    rows = list(csv.DictReader(f))

# функции из урока 2 + превращаем колонки в массивы NumPy
amount = np.array([float(r["amount"].replace(" ", "")) for r in rows])
paid   = np.array([r["status"] == "оплачен" for r in rows])
days   = np.array([float(r["delivery_days"]) if r["delivery_days"] else np.nan for r in rows])
city   = np.array([CITY.get(r["city"].strip().lower(), r["city"].strip()) for r in rows])
cust   = np.array([r["customer_id"] for r in rows])

print(amount.shape, amount.dtype, paid.dtype)

(5463,) float64 bool


In [10]:
# База
#1
pos=amount.argmax()
print('1. Position:',pos)
print('Amount:',amount[pos],'Customer:',cust[pos],'City:',city[pos])
print('Для менеджера: самый дорогой заказ у клиента',cust[pos],'из города',city[pos])

#2
revenue=amount[paid].sum()
unpaid=amount[~paid].sum()
print('2. Paid revenue:',revenue)
print('Unpaid amount:',unpaid)
print('Для менеджера: оплаченная выручка составляет',revenue,'тенге, неоплаченные заказы —',unpaid,'тенге')

#3
shymkent=city=='Шымкент'
shymkent_count=shymkent.sum()
shymkent_share=shymkent.mean()*100
print('3. Shymkent orders:',shymkent_count)
print('Share:',round(shymkent_share,2),'%')
print('Для менеджера: доля заказов из Шымкента составляет',round(shymkent_share,2),'%')


1. Position: 1001
Amount: 9999990.0 Customer: TEST City: Алматы
Для менеджера: самый дорогой заказ у клиента TEST из города Алматы
2. Paid revenue: 101909360.0
Unpaid amount: 7974000.0
Для менеджера: оплаченная выручка составляет 101909360.0 тенге, неоплаченные заказы — 7974000.0 тенге
3. Shymkent orders: 804
Share: 14.72 %
Для менеджера: доля заказов из Шымкента составляет 14.72 %


In [11]:
avg = amount.mean()
print("Средний чек:", round(avg))

Средний чек: 20114


In [12]:
# Стандарт
#4
sale=(paid)&(amount>50000)
loss=(amount[sale]*0.15).sum()
sale_no_test=(paid)&(amount>50000)&(cust!='TEST')
loss_no_test=(amount[sale_no_test]*0.15).sum()
print('4. Discount loss with TEST:',loss)
print('Without TEST:',loss_no_test)
print('Для менеджера: TEST — тестовый клиент с крупными заказами, он завышает оценку потерь от скидки')

#5
print('5. Delivery in 1-2 days by city:')
for ucity in np.unique(city):
    city_mask=city==ucity
    delivery_mask=(days>=1)&(days<=2)
    share=(delivery_mask&city_mask).sum()/city_mask.sum()*100
    print(ucity,round(share,2),'%')
print('Для менеджера: скорость доставки отличается между городами, поэтому общая доля не описывает каждый город')

#6
# Ошибка 1: среднее считается по всем заказам, включая неоплаченные.
# Ошибка 2: TEST — тестовый клиент с крупными заказами, он искажает средний чек.
correct_mask=paid&(cust!='TEST')
avg=amount[correct_mask].mean()
print('6. Correct average check:',round(avg,2))
print('Для менеджера: средний чек посчитан по оплаченным заказам без TEST:',round(avg,2),'тенге')


4. Discount loss with TEST: 5702422.5
Without TEST: 1202427.0
Для менеджера: TEST — тестовый клиент с крупными заказами, он завышает оценку потерь от скидки
5. Delivery in 1-2 days by city:
Актобе 62.22 %
Алматы 64.53 %
Астана 63.53 %
Атырау 64.38 %
Караганда 67.84 %
Шымкент 29.85 %
Для менеджера: скорость доставки отличается между городами, поэтому общая доля не описывает каждый город
6. Correct average check: 14672.39
Для менеджера: средний чек посчитан по оплаченным заказам без TEST: 14672.39 тенге


In [13]:
# Челлендж
#7
import time
start=time.perf_counter()
for repeat in range(100):
    total_loop=0
    for a,p in zip(amount,paid):
        if p:
            total_loop+=a
loop_time=time.perf_counter()-start

start=time.perf_counter()
for repeat in range(100):
    total_numpy=amount[paid].sum()
array_time=time.perf_counter()-start

print('7. Loop sum:',total_loop)
print('NumPy sum:',total_numpy)
print('Loop seconds:',round(loop_time,4))
print('NumPy seconds:',round(array_time,4))
print('NumPy faster by:',round(loop_time/array_time,1),'times')
print('Для менеджера: NumPy позволяет считать выручку быстрее; точное ускорение зависит от компьютера')

#8
np.random.seed(42)
a=np.random.rand(1000)<0.05
b=np.random.rand(1000)<0.06
print('8. Purchases A:',a.sum())
print('Purchases B:',b.sum())

b_better=0
for i in range(100):
    a=np.random.rand(1000)<0.05
    b=np.random.rand(1000)<0.06
    if b.sum()>a.sum():
        b_better+=1
print('B better in:',b_better,'of 100 experiments')
print('Для менеджера: при 1000 клиентах на группу случайность может изменить результат одного теста')


7. Loop sum: 101909360.0
NumPy sum: 101909360.0
Loop seconds: 0.1753
NumPy seconds: 0.002
NumPy faster by: 86.1 times
Для менеджера: NumPy позволяет считать выручку быстрее; точное ускорение зависит от компьютера
8. Purchases A: 54
Purchases B: 58
B better in: 84 of 100 experiments
Для менеджера: при 1000 клиентах на группу случайность может изменить результат одного теста
